# 01 -- Optimizer Fundamentals

This notebook is a hands-on introduction to the seven from-scratch optimizers in this library:
**Gradient Descent, SGD, Adagrad, AdaGrad-Norm, RMSProp, Adam, and L-BFGS**.

We race every optimizer across four classic non-convex benchmark functions --
**Rosenbrock, Himmelblau, Beale, and Rastrigin** -- and visualize their trajectories,
contour plots, and convergence curves.

No PyTorch, no TensorFlow, no autograd: every gradient below is hand-derived NumPy.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
%matplotlib inline

from benchmarks import BENCHMARK_REGISTRY, ROSENBROCK, HIMMELBLAU, BEALE, RASTRIGIN
from optimizers import build_optimizer, OPTIMIZER_REGISTRY
from experiments.benchmark_runner import run_all_optimizers_on_benchmark
from visualization.trajectory import plot_contour, plot_contour_with_trajectories, plot_convergence_curves

print('Available optimizers:', list(OPTIMIZER_REGISTRY))
print('Available benchmark functions:', list(BENCHMARK_REGISTRY))

## The update rules, at a glance

| Optimizer | Update rule |
|---|---|
| Gradient Descent | $w_{t+1} = w_t - \eta \nabla f(w_t)$ |
| SGD (+momentum) | $v_{t+1} = \mu v_t + g_t,\quad w_{t+1} = w_t - \eta v_{t+1}$ |
| Adagrad | $G_t = G_{t-1} + g_t^2,\quad w_{t+1} = w_t - \frac{\eta}{\sqrt{G_t}+\varepsilon} g_t$ |
| AdaGrad-Norm | $b_t^2 = b_{t-1}^2 + \lVert g_t\rVert^2,\quad w_{t+1} = w_t - \frac{\eta}{b_t+\varepsilon} g_t$ |
| RMSProp | $v_t = \rho v_{t-1} + (1-\rho) g_t^2,\quad w_{t+1} = w_t - \frac{\eta}{\sqrt{v_t}+\varepsilon} g_t$ |
| Adam | $m_t,v_t$ bias-corrected 1st/2nd moment EMAs; $w_{t+1} = w_t - \eta \frac{\hat m_t}{\sqrt{\hat v_t}+\varepsilon}$ |
| L-BFGS | two-loop recursion inverse-Hessian approx. + Armijo line search |

See the README and each module's docstring (`optimizers/*.py`) for full derivations, advantages, and disadvantages.

## Static contour plot of each benchmark function

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, fn in zip(axes.ravel(), BENCHMARK_REGISTRY.values()):
    plot_contour(fn, ax=ax)
fig.tight_layout()
plt.show()

## Racing optimizers on the Rosenbrock ('banana') function

The Rosenbrock function has a narrow, curved valley -- a classic ill-conditioning
stress test. Notice how plain SGD/GD zig-zag while Adam and L-BFGS home in
quickly on the global minimum at $(1, 1)$.

In [ ]:
learning_rates = {
    'SGD': 0.0009,
    'Adagrad': 0.5,
    'AdaGrad-Norm': 1.0,
    'RMSProp': 0.01,
    'Adam': 0.05,
    'L-BFGS': 1.0,
}
optimizers = {name: build_optimizer(name, lr=lr) for name, lr in learning_rates.items()}
runs = run_all_optimizers_on_benchmark(ROSENBROCK, optimizers, n_iters=400)

fig1 = plot_contour_with_trajectories(ROSENBROCK, runs)
plt.show()

fig2 = plot_convergence_curves(runs)
plt.show()

for name, run in runs.items():
    print(f'{name:15s} final point={run.final_point.round(4)}  f={run.losses[-1]:.3e}  steps={len(run.trajectory)-1}')

## Try it yourself: Himmelblau, Beale, Rastrigin

Swap `ROSENBROCK` for `HIMMELBLAU`, `BEALE`, or `RASTRIGIN` below and re-run.
Himmelblau has **four** symmetric global minima -- watch different optimizers
(or different starting points) converge to different basins. Rastrigin is
highly multi-modal and a good test of an optimizer's tendency to get stuck
in a local minimum.

In [ ]:
fn = HIMMELBLAU  # try BEALE, RASTRIGIN too
optimizers = {name: build_optimizer(name, lr=lr) for name, lr in {
    'SGD': 0.001, 'Adagrad': 0.5, 'RMSProp': 0.05, 'Adam': 0.2, 'L-BFGS': 1.0,
}.items()}
runs = run_all_optimizers_on_benchmark(fn, optimizers, n_iters=300)
plot_contour_with_trajectories(fn, runs)
plt.show()

## Animated trajectories

`visualization.trajectory.animate_trajectories` produces a `matplotlib.animation.FuncAnimation`
and can save an animated GIF -- see `assets/benchmark_*_animation.gif` for pre-rendered examples,
or generate your own:

```python
from visualization.trajectory import animate_trajectories
animate_trajectories(ROSENBROCK, runs, save_path='../results/rosenbrock_race.gif')
```

Continue to **`02_logistic_regression_a9a.ipynb`** to see these same optimizers
trained on a real binary classification dataset.